# 05 — Induction Heads

Induction heads implement in-context copying: on `[A][B] ... [A]` they
attend to the token *after* the previous `[A]` and predict `[B]`.
We score every head with repeated random sequences (Olsson et al., 2022)
and then causally confirm the top heads by ablating them.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
# Train a tiny model on a synthetic corpus (~30s on CPU).
# The corpus is a seeded word-salad: repetitive enough to learn, varied enough
# that BPE cannot collapse it into a handful of giant tokens.
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
WORDS = ["the", "cat", "dog", "sat", "ran", "on", "to", "mat", "log", "sun"]
CORPUS = " ".join(rng.choice(WORDS) for _ in range(4000))

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=300, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(150)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")

In [ ]:
from kamui.mechinterp import InductionHeadDetector

detector = InductionHeadDetector(model)
scores = detector.score_all_heads(prefix_len=12, n_samples=4)
top = sorted(scores.items(), key=lambda kv: -kv[1])[:3]
print("top heads:", [(f"L{l}H{h}", round(s, 3)) for (l, h), s in top])
detector.plot_scores(scores)

In [ ]:
# Causal check: ablate the single best head and measure in-context loss.
best_head = max(scores, key=scores.get)
delta = detector.ablate_and_measure([best_head], prefix_len=12)
print(f"ablating L{best_head[0]}H{best_head[1]} changes second-half loss by {delta:+.4f}")